# Demo-college — Exploratory Data Analysis op het toernooi van vanmorgen

Dezelfde bewerkingen als in de dia's, maar op data die jullie zelf hebben gemaakt:
het toernooi van week 3. Elke cel hieronder heeft een tegenhanger in de gapminder-versie —
alleen de tabel is anders.

**Je hebt je eigen token nodig**, dezelfde als bij het inleveren.

In [ ]:
# Standaard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# De uitslag ophalen -- exact dezelfde zes regels als Werkcollege 5, Deel 2
API_URL = "https://poker-analytics-api.onrender.com"
STUDENT_ID = "vul_hier_je_studentnummer_in"
TOKEN = "vul_hier_je_token_in"
RONDE = 1

response = requests.get(
    f"{API_URL}/toernooi/3",
    params={"student_id": STUDENT_ID, "ronde": RONDE},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
resultaat = response.json()
print(resultaat["n_bots"], "bots ·", len(resultaat["hand_log"]), "logregels")

💡 Duurt het lang? De eerste aanroep wekt de server; wacht een halve minuut en draai opnieuw.
Geen token bij de hand? De uitslag staat ook als bestand op Brightspace:

```python
import json
resultaat = json.load(open("toernooi_week3.json"))
```

In [ ]:
# Van JSON naar tabel, en meteen de kolom waar het om draait
handen = pd.DataFrame(resultaat["hand_log"])
handen = handen.sort_values(["bot_naam", "simulatie", "hand_nummer"])

# winst = het verschil in stack met de hand ervoor, binnen dezelfde zitting.
# De eerste hand van een zitting heeft geen 'ervoor', dus die meten we vanaf 1000.
handen["winst"] = (handen.groupby(["bot_naam", "simulatie"])["stack"].diff()
                         .fillna(handen["stack"] - 1000))
handen.head()

**Controle vóór alles.** Poker is een nulsom: wat de een wint, verliest een ander.
Komt hier geen 0 uit, dan klopt je groepering niet en is alles hierna onzin.

In [ ]:
handen["winst"].sum()

## Filteren — dezelfde regel als `gapminder[gapminder['continent'] == 'Europe']`

In [ ]:
# Haal uit het log alle handen van één bot
MIJN_BOT = "500959250"   # vul hier je eigen studentnummer in
mijn_handen = handen[handen["bot_naam"] == MIJN_BOT]
mijn_handen

## Groeperen — `groupby('continent')['pop'].sum()`, maar dan per bot

In [ ]:
# Uit het geheel van het toernooi, groeperen op bot en de winst optellen
handen.groupby("bot_naam")["winst"].sum().sort_values(ascending=False)

In [ ]:
# En dit is fout. Waarom?
handen.groupby("bot_naam")["stack"].sum()

## Twee niveaus — `groupby(['continent','year'])`, hier bot en handnummer

In [ ]:
# Groeperen eerst per bot en dan per handnummer: het verloop van de stack
verloop = handen.groupby(["bot_naam", "hand_nummer"])["stack"].mean().reset_index()
verloop.head(10)

In [ ]:
import ipywidgets as widgets
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Maak een widget die alle numerieke waardes op aanvraag laat zien
numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
x_dict = {a : a for a in handen.select_dtypes(include=numerics).columns}

def barchart(x):
    return sns.histplot(x = x, data = handen)

widgets.interact(barchart, x = x_dict)

## `apply()` — een samenvatting per groep

In [ ]:
# Maak een samenvatting van het foldgedrag per bot
handen.groupby("bot_naam")["actie"].apply(lambda s: f"{(s == 'fold').mean():.0%} gefold")

## Wat doet dit stuk code?

Twee manieren om "hoe vaak foldt de klas?" te vragen. Allebei één regel, allebei
zonder foutmelding. Ze schelen twaalf procentpunt.

Welke van de twee is goed — en hoe zou je dat merken als ik het niet zei?

In [ ]:
(handen["actie"] == "fold").mean()

In [ ]:
handen["actie"].value_counts(normalize=True)["fold"]

In [ ]:
# de kolom waar het om draait
handen["aan_zet"].value_counts()

---

## Zelf rekenen (10 min)

Drie vragen over jullie eigen toernooi. Alles wat je nodig hebt staat hierboven al ergens.

In [ ]:
# TODO 1: wie staat er bovenaan? Maak van resultaat["eindstand_per_bot"] een DataFrame
#   met de kolommen 'bot_naam' en 'eindstand', en sorteer aflopend.
#   tip: pd.DataFrame(resultaat["eindstand_per_bot"].items(), columns=[...])


# TODO 2: bereken per bot het percentage fold -- maar nu alleen over de handen
#   waarin hij echt aan zet was. Merge dat met de eindstand.
#   Tel je rijen vóór en ná de merge.


# TODO 3: hoe sterk hangen die twee samen? Gebruik .corr() op de twee kolommen.
#   Wat betekent het teken van dat getal voor jouw bot?

🤔 Kijk naar de bot die bovenaan staat en naar zijn fold-percentage.
Wat zegt dat over de drempels die jij donderdag inlevert?